Eens kijken of de fused mpohamiltoniaan eigenlijk nuttig is.

In [5]:
using LinearAlgebra, MPSKit, TensorKit, MPSKitExperimental
LinearAlgebra.BLAS.set_num_threads(1);

In [6]:
# OpSum-based SU(2) qchem MPO (module in examples/opsum_qchem_su2.jl; checks in examples/opsum/)
include("opsum_qchem_su2.jl"); using .OpSumQChemSU2

In [7]:
file = "N2.CCPVDZ.FCIDUMP"

# ERI is the Electronic Repulsion Integral, K is the two-body term, E0 is an energy shift,
# NORB is the number of orbitals (mps length), NELEC is the number of electrons and MS2 is the total spin.
# fcidump files can contain orbital symmetries, but these are ignored.
(ERI, K, E0, NORB, NELEC, MS2) = parse_fcidump(file);

## OpSum constructie

In [8]:
# same input as the fused MPO below; E0 included
t_opsum = @elapsed ham_opsum = opsum_su2_hamiltonian(E0, K, ERI)
println("OpSum MPO built in $(round(t_opsum; digits = 1)) s")
println("bond dims (dense): ", [dim(right_virtualspace(ham_opsum, i)) for i in 1:NORB])

OpSum MPO built in 44.4 s
bond dims (dense): [17, 128, 263, 462, 725, 1060, 1451, 1914, 2481, 3064, 3711, 4422, 5197, 5924, 5197, 4422, 3711, 3064, 2481, 1962, 1507, 1052, 725, 462, 239, 104, 17, 1]


## Fused constructie

In [9]:
D = 50          # the OpSum MPO uses in-RAM environments; D = 200 likely doesn't fit in 31 GB
sweeps = 10

10

In [10]:

# the second return value contains some metadata which can be ignored
t_fused = @elapsed (ham, _) = fused_quantum_chemistry_hamiltonian(E0, K, ERI, Float64);
println("fused MPO built in $(round(t_fused; digits = 1)) s")

fused MPO built in 125.0 s


In [11]:
psp = Vect[(Irrep[U₁] ⊠ Irrep[SU₂] ⊠ FermionParity)]((0, 0, 0) => 1, (1, 1 // 2, 1) => 1, (2, 0, 0) => 1);

left = Vect[(Irrep[U₁] ⊠ Irrep[SU₂] ⊠ FermionParity)]((-NELEC, MS2 // 2, mod(-NELEC, 2)) => 1);
right = oneunit(left);

virtual = Vect[(Irrep[U₁] ⊠ Irrep[SU₂] ⊠ FermionParity)]((i, s, b) => 1 for i in -NELEC:0, s in 0:(1 // 2):(MS2 // 2 + 1), b in (0, 1));

st = FiniteMPS(rand, Float64, NORB, psp, virtual; left, right);
st0 = copy(st);   # same starting state for both DMRG runs

In [12]:
# Same operator? ⟨st|H|st⟩ on the random starting state (not a ground state, so the value
# itself is meaningless; only the difference matters, expect ~1e-12).
e_fused = real(expectation_value(st, ham, disk_environments(st, ham)))
e_opsum = real(expectation_value(st, ham_opsum))
println("⟨H_opsum⟩ - ⟨H_fused⟩ on a random MPS: ", e_opsum - e_fused)

⟨H_opsum⟩ - ⟨H_fused⟩ on a random MPS: 8.100187187665142e-13


In [13]:

# environments that live on disk, instead of in ram
envs = disk_environments(st, ham);

In [14]:

(st, envs) = find_groundstate(st, ham, DMRG2(; trscheme = truncrank(D), verbosity = 1000, maxiter = sweeps, miniter = sweeps), envs);

println("E = ", real(expectation_value(st, ham, envs)))


┌ Info: DMRG2   1:	obj = -1.080115486976e+02	err = 9.9871016524e-01	time = 1.35 min
└ @ MPSKit /home/maarten/projects/MPSKit.jl/src/algorithms/groundstate/dmrg.jl:197
┌ Info: DMRG2   2:	obj = -1.087646950152e+02	err = 9.9986017615e-01	time = 36.73 sec
└ @ MPSKit /home/maarten/projects/MPSKit.jl/src/algorithms/groundstate/dmrg.jl:197
┌ Info: DMRG2   3:	obj = -1.087829132222e+02	err = 2.6584364393e-03	time = 31.16 sec
└ @ MPSKit /home/maarten/projects/MPSKit.jl/src/algorithms/groundstate/dmrg.jl:197
┌ Info: DMRG2   4:	obj = -1.087961181775e+02	err = 8.5206179238e-04	time = 44.24 sec
└ @ MPSKit /home/maarten/projects/MPSKit.jl/src/algorithms/groundstate/dmrg.jl:197
┌ Info: DMRG2   5:	obj = -1.087978320497e+02	err = 3.3803859688e-04	time = 43.39 sec
└ @ MPSKit /home/maarten/projects/MPSKit.jl/src/algorithms/groundstate/dmrg.jl:197
┌ Info: DMRG2   6:	obj = -1.087990676875e+02	err = 4.2475297314e-04	time = 46.03 sec
└ @ MPSKit /home/maarten/projects/MPSKit.jl/src/algorithms/groundstate/dmrg.

E = -108.80555053168854


┌ Warning: DMRG2 cancel 10:	obj = -1.088055505317e+02	err = 5.6683532987e-04	time = 7.72 min
└ @ MPSKit /home/maarten/projects/MPSKit.jl/src/algorithms/groundstate/dmrg.jl:195


## DMRG met de OpSum MPO

Zelfde startstaat (`st0`), zelfde `D` en `sweeps`. Standaard MPSKit-omgevingen (in RAM), want `disk_environments` is specifiek voor de fused MPO.

In [15]:
(st_opsum, envs_opsum) = find_groundstate(copy(st0), ham_opsum, DMRG2(; trscheme = truncrank(D), verbosity = 1000, maxiter = sweeps, miniter = sweeps));

E_opsum = real(expectation_value(st_opsum, ham_opsum, envs_opsum))
E_fused = real(expectation_value(st, ham, envs))
println("E (OpSum) = ", E_opsum, "    E (fused) = ", E_fused, "    diff = ", E_opsum - E_fused)

┌ Warning: JSON-RPC endpoint has been blocked writing to its peer; outgoing messages are queueing up and will not be delivered until the peer resumes reading
│   blocked_seconds = 120.7
│   queued_messages = 0
└ @ VSCodeServer.JSONRPC /home/maarten/.vscode-oss/extensions/julialang.language-julia-1.243.2-universal/scripts/packages/JSONRPC/src/core.jl:414
┌ Info: DMRG2   1:	obj = -1.080115486980e+02	err = 9.9871016524e-01	time = 3.79 min
└ @ MPSKit /home/maarten/projects/MPSKit.jl/src/algorithms/groundstate/dmrg.jl:197
┌ Info: DMRG2   2:	obj = -1.087646949947e+02	err = 9.9986017333e-01	time = 2.13 min
└ @ MPSKit /home/maarten/projects/MPSKit.jl/src/algorithms/groundstate/dmrg.jl:197
┌ Info: DMRG2   3:	obj = -1.087829132685e+02	err = 2.6584819399e-03	time = 2.00 min
└ @ MPSKit /home/maarten/projects/MPSKit.jl/src/algorithms/groundstate/dmrg.jl:197
┌ Info: DMRG2   4:	obj = -1.087961182524e+02	err = 8.5210920688e-04	time = 2.13 min
└ @ MPSKit /home/maarten/projects/MPSKit.jl/src/algorithms/

E (OpSum) = -108.8055503859439    E (fused) = -108.80555053168854    diff = 1.4574463591543463e-7


┌ Info: ─────────────────────────────────────────────────────────────────────────────────────────────────
│                                         Time                             Allocations
│                           ────────────────────────────────    ──────────────────────────────────
│     Tot / % measured:              1366s / 99.9%                       733GiB / 100.0%
│  ───────────────────────  ────────────────────────────────    ──────────────────────────────────
│  Section          ncalls    time    %tot     avg                alloc    %tot      avg
│ ─────────────────────────────────────────────────────────────────────────────────────────────────
│  sweep                10   1364s  100.0%    136s  ████████     733GiB  100.0%  73.3GiB  ████████
│  ├─ AC2_eigsolve     530   1360s   99.7%   2.57s  ████████     733GiB   99.9%  1.38GiB  ████████
│  ├─ svd_trunc        530   3.32s    0.2%  6.27ms               369MiB    0.0%   713KiB
│  └─ update_AC        530   286ms    0.0%  